# Atividade — Simulador de Gerenciamento de Memória (FIFO e LRU)

Execute as células em ordem. A última célula de código pede a sequência de páginas e a quantidade de frames.

In [1]:
def fifo(paginas, quantidade_frames, mostrar_passos=True):
    """Simula FIFO. Retorna (faltas, hits)."""
    memoria = []  # o início da lista é a página mais antiga
    faltas = 0
    hits = 0

    for pagina in paginas:
        if pagina in memoria:
            hits += 1
            resultado = "HIT"
        else:
            faltas += 1
            resultado = "PAGE FAULT"
            if len(memoria) >= quantidade_frames:
                memoria.pop(0)  # remove a que entrou primeiro
            memoria.append(pagina)

        if mostrar_passos:
            imprimir_passo(pagina, memoria, quantidade_frames, resultado)

    return faltas, hits


def lru(paginas, quantidade_frames, mostrar_passos=True):
    """Simula LRU. Retorna (faltas, hits)."""
    memoria = []  # início = menos recente, fim = mais recente
    faltas = 0
    hits = 0

    for pagina in paginas:
        if pagina in memoria:
            hits += 1
            resultado = "HIT"
            # a página foi usada agora: vira a mais recente
            memoria.remove(pagina)
            memoria.append(pagina)
        else:
            faltas += 1
            resultado = "PAGE FAULT"
            if len(memoria) >= quantidade_frames:
                memoria.pop(0)  # remove a menos recentemente usada
            memoria.append(pagina)

        if mostrar_passos:
            imprimir_passo(pagina, memoria, quantidade_frames, resultado)

    return faltas, hits


def imprimir_passo(pagina, memoria, quantidade_frames, resultado):
    """Mostra uma linha da tabela: Página | Frame 1 | ... | Resultado."""
    frames = [str(p) if i < len(memoria) else "-" for i, p in enumerate(
        memoria + [None] * (quantidade_frames - len(memoria)))]
    colunas = " | ".join(f"{f:^7}" for f in frames)
    print(f"{pagina:^6} | {colunas} | {resultado}")


def imprimir_cabecalho(quantidade_frames):
    colunas = " | ".join(f"{'Frame ' + str(i + 1):^7}" for i in range(quantidade_frames))
    cabecalho = f"{'Página':^6} | {colunas} | Resultado"
    print(cabecalho)
    print("-" * len(cabecalho))


def calcular_taxas(faltas, hits):
    total = faltas + hits
    if total == 0:
        return 0.0, 0.0
    return faltas / total * 100, hits / total * 100


def executar(nome, algoritmo, paginas, quantidade_frames):
    print(f"\n===== {nome} =====")
    imprimir_cabecalho(quantidade_frames)
    faltas, hits = algoritmo(paginas, quantidade_frames, True)
    taxa_fault, taxa_hit = calcular_taxas(faltas, hits)
    print(f"\n{nome}")
    print(f"Page Faults: {faltas}")
    print(f"Hits: {hits}")
    print(f"Taxa de Page Faults: {taxa_fault:.2f}%")
    print(f"Taxa de Hits: {taxa_hit:.2f}%")
    return faltas, hits


def comparar(resultados):
    """resultados: dict nome -> (faltas, hits)."""
    print("\n===== COMPARAÇÃO =====")
    print(f"{'Algoritmo':<10} {'Faults':<8} {'Hits':<8} {'% Faults':<10} {'% Hits':<8}")
    print("-" * 46)
    for nome, (faltas, hits) in resultados.items():
        tf, th = calcular_taxas(faltas, hits)
        print(f"{nome:<10} {faltas:<8} {hits:<8} {tf:<10.2f} {th:<8.2f}")

    f_fifo = resultados["FIFO"][0]
    f_lru = resultados["LRU"][0]
    print()
    if f_fifo < f_lru:
        print(f"Conclusão: o FIFO teve menos page faults ({f_fifo} contra {f_lru}).")
    elif f_lru < f_fifo:
        print(f"Conclusão: o LRU teve menos page faults ({f_lru} contra {f_fifo}).")
    else:
        print(f"Conclusão: empate, os dois tiveram {f_fifo} page faults.")


In [2]:
def ler_entrada():
    while True:
        try:
            entrada = input("Digite as páginas separadas por espaço: ")
            paginas = [int(valor) for valor in entrada.split()]
            if not paginas:
                print("Informe pelo menos uma página.")
                continue
            frames = int(input("Digite a quantidade de frames: "))
            if frames < 1:
                print("A quantidade de frames deve ser pelo menos 1.")
                continue
            return paginas, frames
        except ValueError:
            print("Entrada inválida. Use apenas números inteiros.")


def main():
    paginas, quantidade_frames = ler_entrada()

    print("\nSequência:", paginas)
    print("Frames:", quantidade_frames)

    resultados = {}
    resultados["FIFO"] = executar("FIFO", fifo, paginas, quantidade_frames)
    resultados["LRU"] = executar("LRU", lru, paginas, quantidade_frames)
    comparar(resultados)


main()


Digite as páginas separadas por espaço: 1 2 3 4 1 2 5 1 2 3 4 5
Digite a quantidade de frames: 3

Sequência: [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]
Frames: 3

===== FIFO =====
Página | Frame 1 | Frame 2 | Frame 3 | Resultado
------------------------------------------------
  1    |    1    |    -    |    -    | PAGE FAULT
  2    |    1    |    2    |    -    | PAGE FAULT
  3    |    1    |    2    |    3    | PAGE FAULT
  4    |    2    |    3    |    4    | PAGE FAULT
  1    |    3    |    4    |    1    | PAGE FAULT
  2    |    4    |    1    |    2    | PAGE FAULT
  5    |    1    |    2    |    5    | PAGE FAULT
  1    |    1    |    2    |    5    | HIT
  2    |    1    |    2    |    5    | HIT
  3    |    2    |    5    |    3    | PAGE FAULT
  4    |    5    |    3    |    4    | PAGE FAULT
  5    |    5    |    3    |    4    | HIT

FIFO
Page Faults: 9
Hits: 3
Taxa de Page Faults: 75.00%
Taxa de Hits: 25.00%

===== LRU =====
Página | Frame 1 | Frame 2 | Frame 3 | Resultado
-----

## Exemplo para testar

Quantidade de frames: `3`  
Páginas: `1 2 3 4 1 2 5 1 2 3 4 5`

In [3]:
sequencias = {
    "Clássica": [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2],
    "Com reuso de páginas": [1, 2, 1, 3, 1, 2, 1, 4, 1, 2, 1, 5],
    "Belady": [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5],
}

for nome, seq in sequencias.items():
    f_fifo, h_fifo = fifo(seq, 3, False)
    f_lru, h_lru = lru(seq, 3, False)
    print(f"{nome:<22} FIFO: {f_fifo:<3} faults | LRU: {f_lru:<3} faults")


Clássica               FIFO: 10  faults | LRU: 9   faults
Com reuso de páginas   FIFO: 7   faults | LRU: 5   faults
Belady                 FIFO: 9   faults | LRU: 10  faults


# Questões para análise

**1. HIT x PAGE FAULT.** HIT: a página pedida já está na memória. PAGE FAULT: a página não está na memória e precisa ser carregada (se não houver frame livre, uma página é substituída).

**2. FIFO x LRU.** O FIFO remove a página que entrou primeiro, sem olhar se ela foi usada recentemente. O LRU remove a página que está há mais tempo sem ser usada; por isso um HIT atualiza a ordem no LRU, mas não no FIFO.

**3. Page fault é erro?** Não. É um evento normal da paginação: a página simplesmente ainda não estava na RAM (por exemplo, no primeiro acesso). O sistema operacional trata e o programa continua.

**4. Mais frames sempre reduzem os faults no FIFO?** Não. Na sequência `1 2 3 4 1 2 5 1 2 3 4 5`, o FIFO teve 9 page faults com 3 frames e 10 com 4 frames. É a Anomalia de Belady: aumentar os frames piorou o resultado.

**5. Quando FIFO e LRU diferem?** Quando há páginas reutilizadas com frequência. O LRU mantém essas páginas na memória, e o FIFO pode expulsá-las só porque entraram cedo. Exemplo: em `1 2 1 3 1 2 1 4 1 2 1 5` com 3 frames, FIFO = 7 faults e LRU = 5. Mas o LRU não é sempre melhor: em `1 2 3 4 1 2 5 1 2 3 4 5` com 3 frames, FIFO = 9 e LRU = 10.

**6. Por que o Ótimo costuma ir bem?** Porque ele escolhe sempre a página cujo próximo uso está mais distante no futuro, ou seja, a decisão que menos atrapalha. Gera o menor número possível de faults e serve de referência.

**7. Por que o SO não usa o Ótimo?** Porque ele exigiria conhecer os acessos futuros do processo, e o sistema operacional não consegue prever isso. Por isso usa aproximações baseadas no passado, como o LRU.

**8. Três sequências (3 frames):**

| Sequência | FIFO (faults) | LRU (faults) | Menos faults |
|---|---|---|---|
| `7 0 1 2 0 3 0 4 2 3 0 3 2` | 10 | 9 | LRU |
| `1 2 1 3 1 2 1 4 1 2 1 5` | 7 | 5 | LRU |
| `1 2 3 4 1 2 5 1 2 3 4 5` | 9 | 10 | FIFO |
